In [ ]:
import os
from anthropic import Anthropic
import json
from pathlib import Path

In [ ]:
def getJobData(keyword: str):
    keyword = keyword.lower() # normalize the keyword
    folder = Path("postings") # get filepath to the postings
    results = {}

    for path in folder.glob("*.txt"):
        results[path.name] = []
        text = path.read_text(encoding="utf-8").lower()
        i = 0
        end = len(text)
        # find all instances of the keyword
        while i != -1:
            pos = text.find(keyword, i, end)
            if pos != -1:
                if pos >= 60:
                    results[path.name].append(text[pos-60:pos+61])
                else:
                    results[path.name].append(text[0:pos+61])
                i = pos + len(keyword)
            else: i = -1
    return results

tools = [
    {
        "name": "get_jd_info",
        "description": "Search all of the user's saved job descriptions to get information about a particular skill or qualification needed for them. Ex: Needing to know PyTorch, Kubernetes, C++, etc. Using this tool will return a dictionary where the key is the filename and the value is a list of snippets containing the keyword from the JD. Note that 1 list corresponds to 1 job description, and a list can have 0, 1 or more snippets in it.  If a key has an empty list, this indicates that the keyword was not in the JD.",
        "input_schema": {
            "type": "object",
            "properties": {
                "keyword": {
                    "type": "string",
                    "description": "The skill/qualification you want to learn more about. Note, this tool only returns things based on keyword search, so you'll have to be specific about what you want. For instance, you cannot generalize and ask about AI-based skills and expect to get back an answer that talks about knowing LangGraph, how to build harnesses, etc. This is one case insensitive keyword/phrase per call. You'll get back matches or no matches with the keyword \"AI skills\". Example input for tool use: PyTorch. ",
                },
            },
            "required": ["keyword"],
        },
    }
]

client = Anthropic(
    api_key = os.environ.get("ANTHROPIC_API_KEY")
)

SYSTEM_PROMPT = "You are supposed to help the user go through job postings, identify what she is qualified at, what she lacks, and help her draft outreach messages to companies where her skills mostly match up- you can use your judgement to help decide whether or not she's qualified enough to apply. All the information you have about her is from the profile.md file, which you can access with the get_profile tool. "
messages = [{"role": "user", "content": "I have a few tasks for you: (1) I have a bunch of job descriptions saved and I need to know which of them require/recommend that I know C++. (2) Then based on my profile, I want you to list 5 companies that best match my skill set. (3) From those 5, pick 2 of the best to draft outreach notes to."}]
result = client.messages.create(
    model = "claude-haiku-4-5-20251001",
    max_tokens = 1024,
    system = SYSTEM_PROMPT,
    messages = messages,
    tools = tools,
    tool_choice = {"type": "auto", "disable_parallel_tool_use": True}
)

print("Response 1:", result)

tool_input = None
tool_block = None

if result.stop_reason != "tool_use":
    print("Claude didn't request a tool use. Here is its final response: \n", result.content)

else:

    tool_block = None
    tool_input = None
    for block in result.content:
        if block.type == "tool_use":
            tool_block = block
            if tool_block.name == "get_jd_info":
                tool_input = block.input["keyword"]

    if tool_input == None:
        print("Claude requested an unknown tool: ", tool_block.name)
    else:
        # call tool
        job_info = getJobData(tool_input)

        # need to convert tool result content to a string or list
        job_info = str(job_info)

        # add Claude's message to the list
        messages.append({"role": "assistant", "content": result.content})

        # add tool response to the list
        messages.append({"role":"user",
                            "content": [{
                                "type": "tool_result",
                                "tool_use_id": tool_block.id,
                                "content": job_info
                            }]
                        })

        result = client.messages.create(
            model = "claude-haiku-4-5-20251001",
            max_tokens = 1024,
            messages = messages,
            system = SYSTEM_PROMPT,
            tools = tools,
            tool_choice = {"type": "auto", "disable_parallel_tool_use": True}
        )

        final_ans = ""
        for block in result.content:
            if block.type == "text":
                final_ans = block

        print("Response 2:", final_ans.text)
